# QML — Week 2: Quantum Gates, Simulation, Hardware, and Advantage

**Goal of the week:** know what a gate is, use the Hadamard gate fluently, understand how a classical computer *simulates* qubits, how a real quantum computer *executes* them, and what quantum computers are (and are not) good for.

| # | Topic | ML bridge | Time |
|---|-------|-----------|------|
| 1 | Quantum gates | orthogonal layers, trainable rotation weights | 1.5 h |
| 2 | The Hadamard gate | Walsh–Hadamard rotations in LLM quantization | 1 h |
| 3 | How classical computers simulate qubits | tensor contraction along one axis | 1.5 h |
| 4 | How quantum computers do it under the hood | noisy stochastic hardware | 1 h |
| 5 | Quantum vs. classical: where the benefit is (and isn't) | algorithmic vs. hardware speedup | 1 h |
| 6 | Exercises | | 2 h |
| 7 | Cheat sheet and what's next | | 15 min |

**Prerequisites:** Week 0 (unitaries, tensor product) and Week 1 (state vectors, measurement).

In [ ]:
# pip install numpy matplotlib
import time
from functools import reduce
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(1)

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

---
## 1. Quantum gates

### 1.1 A gate is a unitary matrix

A **quantum gate** is the quantum analogue of a logic gate: an operation that transforms the state. Mathematically it is a **unitary matrix** $U$ ($U^\dagger U=I$):

$$|\psi'\rangle = U|\psi\rangle$$

Consequences (all from Week 0):
- **Norm-preserving** — a valid state stays valid; probabilities still sum to 1.
- **Reversible** — the inverse is $U^\dagger$. Quantum gates never throw information away (unlike classical AND/OR). Measurement is the only irreversible step.
- **Linear** — a gate acts on all amplitudes of a superposition at once.
- **Composition is matrix multiplication, right to left.** "First $A$, then $B$" is $BA$.

### 1.2 The standard single-qubit gates

| Gate | Matrix | What it does |
|---|---|---|
| **X** (NOT) | $\begin{bmatrix}0&1\\1&0\end{bmatrix}$ | bit flip: $\lvert0\rangle\leftrightarrow\lvert1\rangle$ |
| **Z** | $\begin{bmatrix}1&0\\0&-1\end{bmatrix}$ | phase flip: $\lvert1\rangle\to-\lvert1\rangle$ |
| **Y** | $\begin{bmatrix}0&-i\\i&0\end{bmatrix}$ | bit **and** phase flip |
| **S** | $\mathrm{diag}(1, i)$ | quarter-turn phase |
| **T** | $\mathrm{diag}(1, e^{i\pi/4})$ | eighth-turn phase |
| **H** | $\tfrac1{\sqrt2}\begin{bmatrix}1&1\\1&-1\end{bmatrix}$ | Hadamard (section 2) |
| **RX/RY/RZ($\theta$)** | $e^{-i\theta P/2}$ for $P\in\{X,Y,Z\}$ | rotate the Bloch vector by $\theta$ about an axis |

Rotations are the key to QML: $\theta$ is a **continuous, trainable parameter**.

### 1.3 The two-qubit gate: CNOT

$\mathrm{CNOT}$ flips the *target* qubit if the *control* qubit is $|1\rangle$: $|c,t\rangle\mapsto|c,\;t\oplus c\rangle$. Together with single-qubit gates it is **universal** — any unitary can be built from them. It is also what creates **entanglement**:

$$\mathrm{CNOT}\,(H\otimes I)\,|00\rangle=\tfrac1{\sqrt2}(|00\rangle+|11\rangle)$$

In [ ]:
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.diag([1, -1]).astype(complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.diag([1, 1j])
T = np.diag([1, np.exp(1j * np.pi / 4)])

def RX(t): return np.cos(t / 2) * I2 - 1j * np.sin(t / 2) * X
def RY(t): return np.cos(t / 2) * I2 - 1j * np.sin(t / 2) * Y
def RZ(t): return np.cos(t / 2) * I2 - 1j * np.sin(t / 2) * Z

CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]], dtype=complex)

def is_unitary(U): return np.allclose(U.conj().T @ U, np.eye(len(U)))

for name, U in {"X": X, "Y": Y, "Z": Z, "H": H, "S": S, "T": T, "RX(.7)": RX(.7), "RY(.7)": RY(.7), "RZ(.7)": RZ(.7), "CNOT": CNOT}.items():
    print(f"{name:7s} unitary = {is_unitary(U)}")

# reversible: U† undoes U
print("\nU† U |psi> == |psi>:", np.allclose(T.conj().T @ (T @ (0.6 * ket0 + 0.8 * ket1)), 0.6 * ket0 + 0.8 * ket1))

# composition is right-to-left: 'first H, then Z' is  Z @ H
print("first H then Z |0> :", Z @ H @ ket0)

# CNOT on basis states
for b in ["00", "01", "10", "11"]:
    v = np.zeros(4, dtype=complex); v[int(b, 2)] = 1
    out = int(np.argmax(np.abs(CNOT @ v)))
    print(f"CNOT|{b}> = |{out:02b}>")

# Bell state: H on qubit 0, then CNOT
bell = CNOT @ np.kron(H, I2) @ np.kron(ket0, ket0)
print("\nBell:", bell)

### 1.4 Visualization — gates are rotations of the Bloch sphere

Every single-qubit gate is a **rotation** of the Bloch sphere. Below, the same starting state (gray) is sent through six gates (crimson). Then a sweep of the rotation angle of `RX`, `RY`, `RZ`: each traces a circle around its own axis.

In [ ]:
def bloch_vector(psi):
    a, b = psi
    return np.array([2 * (np.conj(a) * b).real, 2 * (np.conj(a) * b).imag, abs(a) ** 2 - abs(b) ** 2])

def draw_sphere(ax):
    u, v = np.mgrid[0:2 * np.pi:40j, 0:np.pi:20j]
    ax.plot_wireframe(np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v), color="lightgray", alpha=0.35, linewidth=0.5)
    for d in np.eye(3):
        ax.plot(*zip(-d, d), color="gray", linewidth=0.8)
    for pos, lab in [((0, 0, 1.28), "|0>"), ((0, 0, -1.32), "|1>"), ((1.32, 0, 0), "|+>"), ((-1.38, 0, 0), "|->"), ((0, 1.3, 0), "|+i>")]:
        ax.text(*pos, lab, ha="center", va="center")
    ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()

def arrow(ax, vec, color, lw=2.5):
    ax.quiver(0, 0, 0, *vec, color=color, linewidth=lw, arrow_length_ratio=0.15)

psi0 = RZ(0.5) @ RY(1.0) @ ket0                        # a generic starting state

fig = plt.figure(figsize=(13, 8))
for i, (name, U) in enumerate({"X": X, "Y": Y, "Z": Z, "H": H, "S": S, "T": T}.items()):
    ax = fig.add_subplot(2, 3, i + 1, projection="3d")
    draw_sphere(ax)
    arrow(ax, bloch_vector(psi0), "gray", 2); arrow(ax, bloch_vector(U @ psi0), "crimson")
    ax.set_title(f"gate {name}:  gray = before, red = after")
plt.tight_layout(); plt.show()

In [ ]:
theta = np.linspace(0, 2 * np.pi, 200)
fig = plt.figure(figsize=(6.5, 6.5)); ax = fig.add_subplot(111, projection="3d"); draw_sphere(ax)
for name, R, c in [("RX", RX, "tab:red"), ("RY", RY, "tab:green"), ("RZ", RZ, "tab:blue")]:
    path = np.array([bloch_vector(R(t) @ psi0) for t in theta])
    ax.plot(path[:, 0], path[:, 1], path[:, 2], color=c, lw=2, label=name + "(θ)")
arrow(ax, bloch_vector(psi0), "black", 2)
ax.legend(loc="upper left"); ax.set_title("sweeping θ: each rotation gate circles its own axis")
plt.tight_layout(); plt.show()

### 1.5 Example for an ML engineer — a rotation gate is a trainable weight

A gate $R_Y(\theta)$ is a $2\times2$ **rotation matrix with one trainable angle**: exactly the "orthogonal layer with a rotation parameter" from Week 0. A QML model is a stack of such gates followed by measuring an observable, so its output is a smooth function of the angles.

For $R_Y(\theta)|0\rangle$ the output $f(\theta)=\langle Z\rangle=\cos\theta$. It is differentiable, and on hardware you can get the derivative from **two extra circuit runs** — the *parameter-shift rule* (Week 3 treats it in depth):

$$\frac{\partial f}{\partial\theta}=\frac{f(\theta+\pi/2)-f(\theta-\pi/2)}{2}$$

In [ ]:
def f(theta):                                        # <Z> after RY(theta)|0>
    psi = RY(theta) @ ket0
    return np.vdot(psi, Z @ psi).real

th = np.linspace(0, 2 * np.pi, 200)
analytic = -np.sin(th)
shift    = np.array([(f(t + np.pi / 2) - f(t - np.pi / 2)) / 2 for t in th])
print("parameter-shift == analytic gradient:", np.allclose(shift, analytic))

plt.plot(th, [f(t) for t in th], label="f(θ) = <Z> = cos θ")
plt.plot(th, shift, "--", label="df/dθ from two circuit runs")
plt.xlabel("θ"); plt.legend(); plt.title("a gate parameter is a differentiable weight"); plt.show()

---
## 2. The Hadamard gate

### 2.1 Definition and basic facts

$$H=\frac1{\sqrt2}\begin{bmatrix}1&1\\1&-1\end{bmatrix},\qquad H|0\rangle=|+\rangle,\quad H|1\rangle=|-\rangle$$

- It turns a definite bit into an **equal superposition** — the standard first step of almost every quantum algorithm.
- It **swaps the Z basis and the X basis**: $H|+\rangle=|0\rangle$, $H|-\rangle=|1\rangle$.
- It is **its own inverse**: $H^2=I$.
- On $n$ qubits, $H^{\otimes n}|0\ldots0\rangle=\frac1{\sqrt{2^n}}\sum_x|x\rangle$ — a uniform superposition over **all $2^n$ bit strings**. In general $H^{\otimes n}|x\rangle=\frac1{\sqrt{2^n}}\sum_y(-1)^{x\cdot y}|y\rangle$.

### 2.2 Answering Week 1's question: $|+\rangle$ vs. $|-\rangle$

In Week 1 we saw that $|+\rangle$ and $|-\rangle$ give identical 50/50 statistics. The Hadamard shows they **are** different: it turns the hidden sign into a visible bit.

$$H|+\rangle=|0\rangle\ (\text{certain}),\qquad H|-\rangle=|1\rangle\ (\text{certain})$$

This is **interference**: the two amplitude paths into each output either add or cancel. Starting from $|0\rangle$ and applying $H$ twice,

$$\text{amplitude of }|1\rangle=\tfrac1{\sqrt2}\cdot\tfrac1{\sqrt2}+\tfrac1{\sqrt2}\cdot\left(-\tfrac1{\sqrt2}\right)=\tfrac12-\tfrac12=0.$$

Insert a $Z$ between the two Hadamards and the second path flips sign, so the amplitude of $|0\rangle$ cancels instead: $HZH=X$.

In [ ]:
plus  = H @ ket0
minus = H @ ket1
print("H|0> =", plus, "  H|1> =", minus)
print("H H = I           :", np.allclose(H @ H, I2))
print("H|+> =", H @ plus, "  H|-> =", H @ minus)
print("H Z H == X        :", np.allclose(H @ Z @ H, X))
print("H X H == Z        :", np.allclose(H @ X @ H, Z))

def n_hadamard(n): return reduce(np.kron, [H] * n)
psi = n_hadamard(3) @ np.eye(8)[0]
print("\nH^3|000> probabilities:", np.abs(psi) ** 2)

### 2.3 Visualization — interference and the Hadamard matrix

Top row: amplitudes through $H,H$ — the $|1\rangle$ amplitude is created and then **cancelled**. Bottom row: $H,Z,H$ — the sign flip makes the $|0\rangle$ amplitude cancel instead. Right: the matrix $H^{\otimes3}$ (entries $\pm1$ before the $1/\sqrt8$ factor) — the sign pattern is $(-1)^{x\cdot y}$.

In [ ]:
def stages(gates):
    s, out = ket0.copy(), [("|0>", ket0.copy())]
    for name, U in gates:
        s = U @ s; out.append((f"after {name}", s.copy()))
    return out

rows = [stages([("H", H), ("H", H)]), stages([("H", H), ("Z", Z), ("H", H)])]
fig = plt.figure(figsize=(14, 6.5))
gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 1, 1.6])
for r, st in enumerate(rows):
    for c, (name, s) in enumerate(st):
        ax = fig.add_subplot(gs[r, c])
        ax.bar(["|0>", "|1>"], s.real, color=["tab:blue" if v >= 0 else "tab:red" for v in s.real])
        ax.axhline(0, color="gray", lw=0.6); ax.set_ylim(-1.05, 1.05); ax.set_title(name, fontsize=10)
        if c: ax.set_yticklabels([])
    for c in range(len(st), 4):
        fig.add_subplot(gs[r, c]).axis("off")
ax = fig.add_subplot(gs[:, 4])
ax.imshow(np.sqrt(8) * n_hadamard(3).real, cmap="bwr", vmin=-1.5, vmax=1.5)
ax.set_title("H⊗H⊗H  (×√8): entries ±1"); ax.set_xlabel("input x"); ax.set_ylabel("output y")
fig.suptitle("top: H,H  → back to |0>      bottom: H,Z,H → |1>      (blue + / red −)", y=1.0)
plt.tight_layout(); plt.show()

### 2.4 Example for an ML engineer — Hadamard rotations flatten outliers (LLM quantization)

The (normalized) Walsh–Hadamard transform is an **orthogonal matrix**: it preserves norms and can be applied fast. Modern LLM quantization methods (e.g. QuIP#, QuaRot) rotate weights/activations with Hadamard matrices before quantizing because a rotation **spreads one huge outlier across all coordinates**, so a uniform low-bit grid wastes fewer levels. Same math as $H^{\otimes n}$ here.

In [ ]:
def quantize(x, bits=4):
    qmax = 2 ** (bits - 1) - 1
    scale = np.abs(x).max() / qmax
    return np.clip(np.round(x / scale), -qmax, qmax) * scale

Hn = n_hadamard(6).real                              # 64 x 64 orthogonal Hadamard matrix
x = rng.normal(size=64); x[5] = 40.0                 # one outlier
y = Hn @ x                                           # rotate

err_plain   = np.linalg.norm(x - quantize(x)) / np.linalg.norm(x)
err_rotated = np.linalg.norm(x - Hn.T @ quantize(y)) / np.linalg.norm(x)
print("norm preserved:", np.linalg.norm(x), np.linalg.norm(y))
print("max/mean |entry|  before:", np.abs(x).max() / np.abs(x).mean(), "  after:", np.abs(y).max() / np.abs(y).mean())
print(f"4-bit relative error  plain: {err_plain:.3f}   Hadamard-rotated: {err_rotated:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
axs[0].bar(range(64), np.abs(x)); axs[0].set_title("before: one outlier dominates")
axs[1].bar(range(64), np.abs(y), color="tab:orange"); axs[1].set_title("after Hadamard: energy spread out")
plt.tight_layout(); plt.show()

---
## 3. How classical computers simulate qubits

### 3.1 The state-vector method

A classical computer cannot *be* a quantum system, but it can **store the state vector** and apply gates as linear algebra. This is what PennyLane's `default.qubit`, Qiskit Aer and every notebook so far do.

| Quantum operation | Classical computation |
|---|---|
| $n$ qubits | array of $2^n$ complex numbers |
| gate on qubit $k$ | tensor contraction along axis $k$ |
| measurement | probabilities $\lvert\psi_i\rvert^2$, then random sampling |

**Naive vs. smart.** Building the full $2^n\times2^n$ matrix $I\otimes\dots\otimes U\otimes\dots\otimes I$ costs $4^n$ memory and time. Instead, reshape $\psi$ to shape $(2,2,\dots,2)$ and contract $U$ with **just axis $k$**: $O(2^n)$ memory and time per gate, no giant matrix.

**The wall.** Memory doubles with every added qubit: 30 qubits ≈ 16 GB, ~45 qubits ≈ petabytes. That is the exponential cost, and the practical limit of exact simulation.

In [ ]:
def zero_state(n):
    psi = np.zeros(2 ** n, dtype=complex); psi[0] = 1
    return psi

def apply_1q(psi, U, k, n):
    # apply 2x2 gate U to qubit k (qubit 0 = most significant, same as np.kron order)
    t = psi.reshape([2] * n)
    t = np.tensordot(U, t, axes=([1], [k]))          # new axis for qubit k appears in front
    return np.moveaxis(t, 0, k).reshape(-1)

def apply_2q(psi, U, c, t_, n):
    # apply 4x4 gate U to qubits (c, t_)
    t = psi.reshape([2] * n)
    t = np.tensordot(U.reshape(2, 2, 2, 2), t, axes=([2, 3], [c, t_]))
    return np.moveaxis(t, [0, 1], [c, t_]).reshape(-1)

def sample(psi, shots):
    p = np.abs(psi) ** 2
    return rng.choice(len(psi), size=shots, p=p / p.sum())

# --- check against the naive kron construction on 3 qubits
n = 3
full_H1 = reduce(np.kron, [I2, H, I2])
psi = np.zeros(8, dtype=complex); psi[5] = 1
print("matches naive kron:", np.allclose(apply_1q(psi, H, 1, n), full_H1 @ psi))

# --- Bell state with the mini-simulator
psi = zero_state(2)
psi = apply_1q(psi, H, 0, 2)
psi = apply_2q(psi, CNOT, 0, 1, 2)
print("Bell via simulator:", psi)

In [ ]:
# GHZ state on 20 qubits: (|0...0> + |1...1>)/sqrt2.  2^20 ~ 1 million amplitudes
n = 20
t0 = time.perf_counter()
psi = apply_1q(zero_state(n), H, 0, n)
for k in range(n - 1):
    psi = apply_2q(psi, CNOT, k, k + 1, n)
dt = time.perf_counter() - t0

nz = np.flatnonzero(np.abs(psi) > 1e-9)
print(f"{n} qubits, {len(psi):,} amplitudes, {n} gates in {dt:.2f}s")
print("non-zero basis states:", [format(i, f'0{n}b') for i in nz], "  amplitudes:", psi[nz])
print("10 measurement shots :", [format(i, f'0{n}b') for i in sample(psi, 10)][:3], "...")

### 3.2 Visualization — the exponential wall

Left: time to apply one gate. The naive $2^n\times2^n$ matrix grows like $4^n$, the tensor-contraction method like $2^n$ — both exponential, but the second reaches much further. Right: memory of the state vector.

In [ ]:
def time_it(fn, reps=3):
    best = 1e9
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

ns_naive, t_naive = list(range(2, 12)), []
for n in ns_naive:
    psi = rng.normal(size=2 ** n) + 0j
    t_naive.append(time_it(lambda: reduce(np.kron, [H] + [I2] * (n - 1)) @ psi, reps=2))

ns_fast, t_fast = list(range(2, 23, 2)), []
for n in ns_fast:
    psi = rng.normal(size=2 ** n) + 0j
    t_fast.append(time_it(lambda: apply_1q(psi, H, 0, n)))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].semilogy(ns_naive, t_naive, "o-", label="naive 2ⁿ×2ⁿ matrix  (~4ⁿ)")
axs[0].semilogy(ns_fast, t_fast, "s-", label="tensor contraction  (~2ⁿ)")
axs[0].set_xlabel("qubits n"); axs[0].set_ylabel("seconds per gate"); axs[0].legend(); axs[0].set_title("time to apply one gate")

ns = np.arange(1, 51)
axs[1].semilogy(ns, (2.0 ** ns) * 16 / 1e9)
axs[1].axhline(16, color="gray", ls="--", label="16 GB laptop"); axs[1].axhline(1e6, color="tab:red", ls=":", label="1 PB")
axs[1].set_xlabel("qubits n"); axs[1].set_ylabel("state-vector memory (GB)"); axs[1].legend(); axs[1].set_title("memory wall")
plt.tight_layout(); plt.show()

> **Shortcuts exist, but not for everything.**
> - *Stabilizer simulation* (Gottesman–Knill): circuits made only of H, S, CNOT (Clifford gates) can be simulated in polynomial time — they are not where quantum advantage lives.
> - *Tensor-network simulation*: cheap when entanglement is low; cost explodes as entanglement grows.
> - *Sampling-based tricks* for approximate answers.
>
> For generic, highly entangled circuits at 50+ qubits, no known classical method avoids exponential cost.

### 3.3 Example for an ML engineer — a gate is a linear layer along one tensor axis

`np.tensordot(U, psi, axes=([1],[k]))` is the same operation as applying a dense layer **along one dimension** of a tensor (as in MLP-Mixer or an `einsum('ij,...j...->...i...')`). A quantum circuit simulation is a sequence of such mixing layers on a $2\times2\times\dots\times2$ tensor — which is why it runs well on GPUs, and why GPU simulators (cuQuantum, PennyLane Lightning-GPU) are the workhorse for QML research today.

---
## 4. How quantum computers do it under the hood

### 4.1 A quantum computer does not store $2^n$ numbers

The key conceptual difference: a classical simulator **writes down** the amplitudes. A quantum computer **is** a physical system whose state *is* the superposition. Nothing is stored or looked up; the physics does the evolution.

| Layer | What happens |
|---|---|
| **Qubit** | a two-level physical system (see table) |
| **Gate** | a **control pulse** (microwave or laser) turns the system's Hamiltonian on for a set time; the state evolves by $U=e^{-iHt/\hbar}$ |
| **Circuit** | a timed sequence of pulses |
| **Measurement** | readout (e.g. microwave resonator, fluorescence) returns one classical bit per qubit; the state collapses |
| **Shots** | repeat the whole run thousands of times to build up statistics |

Common hardware platforms:

| Platform | Qubit | Gates by |
|---|---|---|
| Superconducting circuits (IBM, Google) | transmon circuit at ~15 mK | microwave pulses |
| Trapped ions (Quantinuum, IonQ) | internal states of ions in a trap | lasers |
| Neutral atoms (QuEra, Pasqal, Atom Computing) | atoms held by optical tweezers | lasers |
| Photonics (PsiQuantum, Xanadu) | photon modes | interferometers, detectors |

**From your code to pulses:** the circuit is *transpiled* — rewritten into the device's **native gate set** and routed so two-qubit gates only touch physically connected qubits (extra SWAPs cost depth). Then each native gate becomes a calibrated pulse.

### 4.2 Gate = pulse: the Rabi oscillation

Drive a qubit with Hamiltonian $H=\tfrac\Omega2X$ for time $t$:

$$U(t)=e^{-i\,\Omega t\,X/2}=R_X(\Omega t)$$

So the **pulse duration sets the rotation angle**: a "π-pulse" ($\Omega t=\pi$) is an X gate, a "π/2-pulse" makes $|+i\rangle$-type states. Measuring after pulses of increasing length gives $P(1)=\sin^2(\Omega t/2)$ — the *Rabi oscillation* — the first calibration experiment on any new qubit.

### 4.3 The catch: noise
Real qubits leak information to the environment (**decoherence**): energy relaxation ($T_1$) and dephasing ($T_2$). Below we add energy relaxation to the Rabi experiment with a simple master equation (Lindblad form), and sample finite shots as hardware would.

In [ ]:
sm = np.array([[0, 1], [0, 0]], dtype=complex)          # sigma_minus = |0><1| : decay |1> -> |0>

def rhs(rho, Hh, gamma):
    comm = -1j * (Hh @ rho - rho @ Hh)
    L, Ld = sm, sm.conj().T
    return comm + gamma * (L @ rho @ Ld - 0.5 * (Ld @ L @ rho + rho @ Ld @ L))

def rabi(Omega, gamma, t_max, dt=0.002):
    Hh = 0.5 * Omega * X
    rho = np.array([[1, 0], [0, 0]], dtype=complex)
    ts, p1 = [0.0], [0.0]
    for step in range(int(t_max / dt)):
        k1 = rhs(rho, Hh, gamma); k2 = rhs(rho + dt / 2 * k1, Hh, gamma)
        k3 = rhs(rho + dt / 2 * k2, Hh, gamma); k4 = rhs(rho + dt * k3, Hh, gamma)
        rho = rho + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        ts.append((step + 1) * dt); p1.append(rho[1, 1].real)
    return np.array(ts), np.array(p1)

Omega = 2 * np.pi                                       # one full Rabi cycle per time unit (e.g. per microsecond)
t_ideal, p_ideal = rabi(Omega, 0.0, 4.0)
t_noisy, p_noisy = rabi(Omega, 0.6, 4.0)

print("ideal matches sin^2(Omega t/2):", np.allclose(p_ideal, np.sin(Omega * t_ideal / 2) ** 2, atol=1e-4))

pts = np.linspace(0.1, 3.9, 25)
shots = 200
meas = [rng.binomial(shots, np.interp(t, t_noisy, p_noisy)) / shots for t in pts]

plt.figure(figsize=(8, 4))
plt.plot(t_ideal, p_ideal, label="ideal  sin²(Ωt/2)")
plt.plot(t_noisy, p_noisy, label="with energy relaxation (T₁)")
plt.errorbar(pts, meas, yerr=np.sqrt(np.array(meas) * (1 - np.array(meas)) / shots), fmt="k.", label=f"'measured' ({shots} shots)")
plt.axvline(0.5, color="gray", ls=":"); plt.text(0.52, 0.05, "π-pulse = X gate", color="gray")
plt.xlabel("pulse duration t"); plt.ylabel("P(measure 1)"); plt.legend(); plt.title("Rabi oscillation: pulse length = rotation angle")
plt.tight_layout(); plt.show()

### 4.4 Visualization — why noise limits circuit depth

If every gate succeeds with probability $1-\varepsilon$, a circuit with $N$ gates succeeds roughly with $(1-\varepsilon)^N$. Today's two-qubit gate errors are of order $10^{-3}$–$10^{-2}$ (varies by platform and device), so useful depth is limited to hundreds to thousands of gates. **Quantum error correction** encodes one *logical* qubit into many *physical* qubits (commonly estimated at hundreds to thousands, depending on error rates and code) so that errors can be detected and removed — this is the central engineering challenge of the field.

In [ ]:
N = np.logspace(0, 4.5, 200)
for eps, c in [(1e-4, "tab:green"), (1e-3, "tab:blue"), (1e-2, "tab:red")]:
    plt.semilogx(N, (1 - eps) ** N, color=c, label=f"gate error ε = {eps:g}")
plt.axhline(0.5, color="gray", ls=":")
plt.xlabel("number of gates N"); plt.ylabel("rough success probability  (1−ε)ᴺ"); plt.legend()
plt.title("without error correction, depth is limited by ε"); plt.show()

### 4.5 Example for an ML engineer — hardware is a noisy stochastic function evaluator

From the programmer's side, a quantum device is a **black-box function that returns noisy Monte-Carlo estimates**:

- **Shot noise** — each expectation value $\langle O\rangle$ is an average of finite samples, error $\propto1/\sqrt{\text{shots}}$ (Week 1).
- **Hardware noise** — a systematic bias on top (decoherence, gate errors, readout errors).
- **Training** a QML model on hardware is therefore **SGD with noisy gradients** (via parameter shift), and variance reduction / error mitigation matter just as they do for noisy analog accelerators. Shots cost wall-clock time, so the number of circuit evaluations is the real budget — much like number of function evaluations in black-box optimization.

---
## 5. Quantum vs. classical: where the benefit is (and isn't)

### 5.1 What quantum computers are *not*

The popular story — "a quantum computer tries all answers at once" — is wrong. A circuit does create a superposition of all $2^n$ inputs, but **measurement returns only one random outcome**. A speedup exists only when the algorithm arranges **interference** so that wrong answers cancel and the right one is amplified (as in $HH=I$). That choreography exists for specific problem structures, not in general. Notably, NP-complete problems are *not* believed to be efficiently solvable on quantum computers.

### 5.2 Where benefit is known or expected

| Problem | Quantum algorithm | Speedup vs. best known classical | Status |
|---|---|---|---|
| Unstructured search over $N$ items | **Grover** | quadratic: $\sqrt N$ vs. $N$ queries | proven optimal; modest gain |
| Integer factoring / discrete log | **Shor** | super-polynomial (best classical is sub-exponential, not proven optimal) | needs large fault-tolerant machines |
| Simulating quantum systems (chemistry, materials) | Hamiltonian simulation, phase estimation | believed exponential | expected the most natural application; not yet shown at useful scale |
| Random circuit sampling | direct execution | beyond-classical demonstrations claimed (e.g. Google 2019) | useless task; classical methods keep improving, so claims stay contested |
| Linear algebra / recommendation (HHL-style) | HHL etc. | exponential on paper | **caveats**: data loading and readout; several were "dequantized" by classical algorithms |
| **Quantum machine learning** | variational circuits, quantum kernels | **no proven practical advantage** | open research question |

### 5.3 Why the speedups are fragile

1. **Data loading.** Encoding $N$ classical numbers into amplitudes takes $O(N)$ operations — often erasing the speedup before the computation starts.
2. **Readout.** $n$ qubits return only $n$ classical bits per shot; reading all $2^n$ amplitudes would need exponentially many shots.
3. **Noise.** Today's devices (NISQ: noisy, intermediate-scale) can run only shallow circuits (section 4.4).
4. **Competing classical algorithms.** Whenever a quantum speedup is claimed, classical researchers try to match it.

### 5.4 Visualization — Grover's search with our own simulator

Grover's algorithm is the cleanest example of interference doing real work. Start in the uniform superposition (Hadamards). Repeat: **oracle** flips the sign of the marked item; **diffusion** reflects all amplitudes about their mean — together they rotate the state toward the marked item. After $k$ rounds, $P(\text{marked})=\sin^2((2k+1)\theta)$ with $\sin\theta=1/\sqrt N$. The optimum is about $\frac\pi4\sqrt N$ rounds — **overshoot and it gets worse**.

In [ ]:
def grover_probs(n, marked, iters):
    N = 2 ** n
    psi = zero_state(n)
    for k in range(n): psi = apply_1q(psi, H, k, n)             # uniform superposition
    probs = [abs(psi[marked]) ** 2]
    for _ in range(iters):
        psi = psi.copy(); psi[marked] *= -1                      # oracle: phase-flip the marked item
        for k in range(n): psi = apply_1q(psi, H, k, n)
        psi = psi.copy(); psi[0] *= -1                           # reflect about |0...0>
        for k in range(n): psi = apply_1q(psi, H, k, n)
        psi = -psi                                               # global phase: makes diffusion 2|s><s| - I
        probs.append(abs(psi[marked]) ** 2)
    return np.array(probs)

n, marked = 4, 11
N = 2 ** n
k = np.arange(0, 11)
sim = grover_probs(n, marked, 10)
theta = np.arcsin(1 / np.sqrt(N))
theory = np.sin((2 * k + 1) * theta) ** 2

print("simulated probabilities match theory:", np.allclose(sim, theory))
print(f"optimal rounds ≈ {np.pi / 4 * np.sqrt(N):.1f}; P(marked) after 3 rounds = {sim[3]:.3f}  (random guess = {1 / N:.3f})")

plt.figure(figsize=(7, 4))
plt.plot(k, theory, "-", label="theory sin²((2k+1)θ)")
plt.plot(k, sim, "o", label="our simulator")
plt.axhline(1 / N, color="gray", ls=":", label="classical guess 1/N")
plt.xlabel("Grover rounds k"); plt.ylabel("P(marked item)"); plt.legend(); plt.title(f"Grover on N={N} items: amplify, then overshoot")
plt.show()

### 5.5 Example for an ML engineer — algorithmic vs. hardware speedup

A GPU gives a **constant-factor hardware speedup** (say 50×) on the *same* algorithm. A quantum algorithm like Grover changes the **scaling**: $\sqrt N$ instead of $N$ queries. The gap widens with problem size — but quadratic is far weaker than exponential, and the quantum machine runs far slower per step than a CPU today. Plot the expected number of oracle queries to find one item among $N$:

In [ ]:
Ns = np.logspace(2, 14, 200)
classical = Ns / 2                      # expected random-search queries
grover = np.pi / 4 * np.sqrt(Ns)

plt.figure(figsize=(7, 4))
plt.loglog(Ns, classical, label="classical random search  ≈ N/2")
plt.loglog(Ns, grover, label="Grover  ≈ (π/4)√N")
plt.xlabel("search space size N"); plt.ylabel("oracle queries"); plt.legend()
plt.title("quadratic speedup: real, but not exponential"); plt.show()

for N_ in [1e4, 1e8, 1e12]:
    print(f"N = {N_:.0e}:  classical ≈ {N_ / 2:.1e}   Grover ≈ {np.pi / 4 * np.sqrt(N_):.1e}   ratio ≈ {N_ / 2 / (np.pi / 4 * np.sqrt(N_)):.1e}")

> **Take-away for QML.** There is no proof that quantum models learn better than classical ones on classical data. The honest research questions are: *which* data/problem structures give an advantage, and can the loading and noise bottlenecks be overcome? We return to this critically after variational circuits and kernels (Weeks 3–4).

---
## 6. Exercises

Do them on paper first, then verify in code.

**Exercise 1 — gate identities.** Verify numerically that $HXH=Z$, $HZH=X$ and $HYH=-Y$. Why does conjugating by $H$ swap $X\leftrightarrow Z$ (think: it swaps the Z and X bases)?

In [ ]:
# TODO

**Exercise 2 — rotations.** Show that $R_Y(\pi)|0\rangle=|1\rangle$ up to a sign, and that $R_Z(\theta)$ never changes the measurement probabilities of $|0\rangle$ or $|1\rangle$ but does change $|+\rangle$. Verify with `bloch_vector`.

In [ ]:
# TODO

**Exercise 3 — circuit order.** Compute $ZH|0\rangle$ and $HZ|0\rangle$. They differ. Which one produces $|-\rangle$? Explain using right-to-left composition.

In [ ]:
# TODO

**Exercise 4 — build GHZ.** With `apply_1q`/`apply_2q`, build the 3-qubit GHZ state $(|000\rangle+|111\rangle)/\sqrt2$ and verify its probabilities. Then sample 1000 shots and check that only `000` and `111` appear.

In [ ]:
# TODO

**Exercise 5 — memory wall.** How much RAM does a 35-qubit state vector (complex128) need? A 40-qubit one? What if you use complex64?

In [ ]:
# TODO

**Exercise 6 — π-pulse.** In the Rabi experiment with $\Omega=2\pi$, find the pulse duration of an X gate and of an $R_X(\pi/2)$ gate. Which measurement probabilities do they produce?

In [ ]:
# TODO

**Exercise 7 — Grover.** For $n=5$ ($N=32$) find the optimal number of rounds and the success probability. What happens at twice that number of rounds?

In [ ]:
# TODO

**Exercise 8 (ML link).** Repeat the Hadamard-quantization experiment with 3 outliers instead of 1, and with 2-bit instead of 4-bit quantization. Plot relative error vs. number of outliers with and without rotation.

In [ ]:
# TODO

---
## 7. Cheat sheet and what's next

| Concept | One-liner |
|---|---|
| Gate | unitary matrix: norm-preserving, reversible, composed right-to-left |
| X, Z, Y | Pauli gates: bit flip, phase flip, both |
| RX/RY/RZ($\theta$) | Bloch-sphere rotations — the trainable parameters of QML |
| CNOT | controlled-NOT; with 1-qubit gates it is universal and creates entanglement |
| Hadamard | $\lvert0\rangle\to\lvert+\rangle$, swaps Z and X bases, $H^2=I$, $H^{\otimes n}$ = uniform superposition |
| Interference | amplitudes add, then get squared — $HH=I$ cancels, $HZH=X$ |
| Classical simulation | $2^n$ amplitudes, contract each gate along one axis; exponential wall at ~45 qubits |
| Real hardware | physical qubits; gates = timed pulses $e^{-iHt}$; measurement = 1 bit per qubit per shot; noise limits depth |
| Quantum advantage | specific structured problems (factoring, simulation, search (quadratic)); not general; not yet shown for ML |

**Next — Week 3: variational circuits and data encoding.** Parameterized gates become a trainable model: angle/amplitude encoding, observables as outputs, the parameter-shift rule, and your first QNN in PennyLane.